# OptiMulti-Video: High-Performance Multimodal Attention (T4 Gpus)

This notebook demonstrates the **OptiMulti-Video** project, featuring:
1. **Custom CUDA Kernel**: Fused Normalization & Projection.
2. **Distributed Training**: FSDP on T4 GPUs.

## 1. Environment Setup

In [ ]:
!nvidia-smi

## 2. Get the Code
Running this on Colab requires the source code. 
**Option A (Recommended)**: Clone your GitHub repository.
**Option B**: Upload the `src/`, `model/`, `training/` folders and `setup.py` manually to the Files tab.

In [ ]:
# OPTION A: Clone your repo
!git clone https://github.com/Ferasman979/OptiMulti-Video.git
%cd OptiMulti-Video

## 3. Compile Custom CUDA Kernel
We use `pip install .` to compile the C++ extension on the attached GPU.

In [ ]:
!pip install -v .

## 4. Run FSDP Distributed Training
We spawn 2 processes (if 2 GPUs are available) to train the model.

In [ ]:
!python training/train_fsdp.py

## 5. Verify Custom Kernel
Let's run a quick numerical check to ensure our CUDA kernel matches PyTorch.

In [ ]:
import torch
import optimulti_fusion_cuda

if torch.cuda.is_available():
    device = torch.device('cuda')
    a = torch.randn(16, 128, 768, device=device)
    b = torch.randn(16, 128, 768, device=device)
    out_cuda = torch.zeros_like(a)
    
    # Custom Op
    optimulti_fusion_cuda.fused_add_layernorm(a, b, out_cuda, 1e-5)
    
    # PyTorch Ref
    out_ref = torch.nn.functional.layer_norm(a + b, (768,), eps=1e-5)
    
    diff = (out_cuda - out_ref).abs().max().item()
    print(f"Max Difference: {diff}")
    assert diff < 1e-3, "Kernel mismatch!"
    print("verification Passed!")
else:
    print("No GPU available for verification.")